# Sizing style 2026-09 — account-level risk vs per-trade stops (AUDIT + policy decision)

Pre-registration: [README.md](README.md) (frozen before any run). Write-up: [findings.md](findings.md).

| id | question |
|---|---|
| S1 | exit policy on identical entries: shipped stop vs time-only vs target-only vs 3× catastrophe stop |
| S2 | sizing rule on the shipped trades (fixed-R on fixed capital / on equity / fixed notional) and the ruin table |
| S3 | minute-by-minute cross-margin liquidation walk per bot and pooled; pre-registered decision |

Re-run order: `run_s1_exit_policies.py` → `run_s2_sizing_rules.py` → `run_s3_liquidation.py`.


## S1 — exit policies

In [1]:
%run run_s1_exit_policies.py


CHENTO_BTC (cost 9.6 bp, sizing (2.0, 3.0))
                       n  mean_r    win  worst_r   h1_r   h2_r  mean_pct  mtm_maxdd_pct  pct_per_yr    mar  notional_x    d_r
P0_shipped         101.0   0.800  0.406   -1.000  1.301  0.289     1.491         -9.912      26.963  2.720       1.248  0.000
P1_time_only       101.0   0.779  0.525   -5.774  1.580 -0.038     1.460        -14.899      25.996  1.745       1.248 -0.021
P1b_target_only    101.0   0.768  0.535   -5.774  1.388  0.136     1.438        -15.934      26.032  1.634       1.248 -0.032
P2_catastrophe_3x  101.0   0.714  0.525   -3.000  1.347  0.068     1.332        -15.904      24.107  1.516       1.248 -0.086



CHENTO_ETH (cost 10.0 bp, sizing (2.0, 3.0))
                      n  mean_r    win  worst_r   h1_r   h2_r  mean_pct  mtm_maxdd_pct  pct_per_yr    mar  notional_x    d_r
P0_shipped         77.0   0.653  0.429   -1.000  0.979  0.318     1.147        -10.745      16.521  1.538       0.981  0.000
P1_time_only       77.0   0.595  0.494   -3.373  1.231 -0.057     0.953        -20.434      13.384  0.655       0.981 -0.058
P1b_target_only    77.0   0.466  0.506   -3.373  1.005 -0.087     0.773        -21.675      10.977  0.506       0.981 -0.187
P2_catastrophe_3x  77.0   0.328  0.506   -3.000  0.761 -0.117     0.496        -25.250       7.219  0.286       0.981 -0.325



SHORT_SQUEEZE (cost 9.3 bp, sizing (1.0, 3.0))
                      n  mean_r    win  worst_r   h1_r   h2_r  mean_pct  mtm_maxdd_pct  pct_per_yr    mar  notional_x    d_r
P0_shipped         71.0   0.481  0.451   -1.000  0.388  0.576     0.085         -7.881       1.504  0.191       2.497  0.000
P1_time_only       71.0   1.164  0.634   -7.819  0.504  1.843     0.497        -12.634       8.772  0.694       2.497  0.683
P1b_target_only    71.0   0.665  0.662   -7.819  0.372  0.967     0.232        -12.379       4.084  0.330       2.497  0.185
P2_catastrophe_3x  71.0   0.481  0.606   -3.000  0.481  0.482     0.103        -12.899       1.812  0.140       2.497  0.001



SQUEEZE_BULL (cost 6.7 bp, sizing (1.0, 3.0))
                       n  mean_r    win  worst_r   h1_r   h2_r  mean_pct  mtm_maxdd_pct  pct_per_yr    mar  notional_x    d_r
P0_shipped         122.0   0.334  0.582   -1.000  0.256  0.412     0.301         -4.399       8.199  1.864         0.5  0.000
P1_time_only       122.0   0.566  0.648   -4.171  0.556  0.575     0.533         -3.989      14.731  3.693         0.5  0.232
P1b_target_only    122.0   0.526  0.689   -4.171  0.544  0.507     0.492         -3.315      13.560  4.090         0.5  0.192
P2_catastrophe_3x  122.0   0.422  0.672   -3.000  0.486  0.358     0.389         -5.274      10.697  2.028         0.5  0.088


ADX P0_shipped_T2 {'n': 27, 'mean_net_pct': 17.152, 'worst_pct': -10.1, 'win': 0.556, 'first_half_pct': 22.984, 'second_half_pct': 10.871, 'cagr_pct': 40.27, 'mtm_maxdd_pct': -38.102, 'mar': 1.057, 'sharpe': 1.093, 'reasons': {'ADX<20': 15, 'SL': 6, 'ATR_trail': 6}}
ADX P1_signal_only {'n': 27, 'mean_net_pct': 21.299, 'worst_pct': -28.873, 'win': 0.667, 'first_half_pct': 29.391, 'second_half_pct': 12.584, 'cagr_pct': 47.923, 'mtm_maxdd_pct': -55.143, 'mar': 0.869, 'sharpe': 1.085, 'reasons': {'ADX<20': 27}}
ADX P2_catastrophe_30 {'n': 27, 'mean_net_pct': 20.742, 'worst_pct': -30.1, 'win': 0.667, 'first_half_pct': 28.406, 'second_half_pct': 12.489, 'cagr_pct': 44.737, 'mtm_maxdd_pct': -48.16, 'mar': 0.929, 'sharpe': 1.066, 'reasons': {'ADX<20': 25, 'SL': 2}}


## S2 — sizing rules and ruin table

In [2]:
%run run_s2_sizing_rules.py


CHENTO_BTC
                       final_equity  cagr_pct  maxdd_pct   mar  worst_trade_pct_equity      n  years
fixed_R_fixed_capital      25060.63     17.97      -9.07  1.98                   -2.02  101.0   5.56
fixed_R_on_equity          39544.61     28.05     -22.19  1.26                   -2.29  101.0   5.56
fixed_notional             22905.95     16.07     -11.87  1.35                  -11.87  101.0   5.56

CHENTO_ETH
                       final_equity  cagr_pct  maxdd_pct   mar  worst_trade_pct_equity     n  years
fixed_R_fixed_capital      18833.68     12.30     -10.15  1.21                   -2.12  77.0   5.46
fixed_R_on_equity          22398.35     15.93     -17.47  0.91                   -2.26  77.0   5.46
fixed_notional             17963.34     11.33     -12.20  0.93                   -4.51  77.0   5.46

SHORT_SQUEEZE
                       final_equity  cagr_pct  maxdd_pct   mar  worst_trade_pct_equity     n  years
fixed_R_fixed_capital      10606.77      1.47      -7.19 


liquidating adverse move by gross exposure: {'0.5x': 199.5, '1x': 99.5, '2x': 49.5, '3x': 32.83333333333333, '5x': 19.5, '10x': 9.5}
worst moves: {'BTC': {'1h': {'worst_drop_pct': -24.77860513782457, 'drop_at': '2020-03-12 10:48', 'worst_rise_pct': 39.523760420715305, 'rise_at': '2020-03-13 02:45'}, '6h': {'worst_drop_pct': -36.85926544240401, 'drop_at': '2020-03-13 02:16', 'worst_rise_pct': 48.064714856443324, 'rise_at': '2020-03-13 08:05'}, '24h': {'worst_drop_pct': -51.31610619469027, 'drop_at': '2020-03-13 02:16', 'worst_rise_pct': 57.45096017323572, 'rise_at': '2020-03-13 13:34'}, '72h': {'worst_drop_pct': -53.58780218431709, 'drop_at': '2020-03-13 02:16', 'worst_rise_pct': 57.45096017323572, 'rise_at': '2020-03-13 13:34'}}, 'ETH': {'1h': {'worst_drop_pct': -30.99667409816892, 'drop_at': '2021-05-19 13:10', 'worst_rise_pct': 41.896984924623105, 'rise_at': '2020-03-13 03:29'}, '6h': {'worst_drop_pct': -37.413579797685756, 'drop_at': '2020-03-13 02:16', 'worst_rise_pct': 60.4651162

## S3 — liquidation walk and decision

In [3]:
%run run_s3_liquidation.py

P0_shipped CHENTO_BTC {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.2948, 'at': '2023-10-08 13:12', 'max_gross_x': 3.3354, 'min_equity': 9661.4404, 'worst_mae': 0.1134, 'final_equity': 25060.6348, 'open_minutes': 206192}


P0_shipped CHENTO_ETH {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.3138, 'at': '2023-10-13 20:54', 'max_gross_x': 3.1372, 'min_equity': 9395.3573, 'worst_mae': 0.0822, 'final_equity': 18833.6789, 'open_minutes': 173743}


P0_shipped SHORT_SQUEEZE {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.3292, 'at': '2023-05-13 07:15', 'max_gross_x': 2.9922, 'min_equity': 9847.015, 'worst_mae': 0.0189, 'final_equity': 10606.7742, 'open_minutes': 8922}


P0_shipped SQUEEZE_BULL {'breach_minutes': 0, 'episodes': 0, 'min_distance': 1.0503, 'at': '2023-04-03 21:00', 'max_gross_x': 0.9476, 'min_equity': 9825.3841, 'worst_mae': 0.0385, 'final_equity': 13666.8851, 'open_minutes': 186049}


P0_shipped POOLED $50k {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.9376, 'at': '2023-12-03 19:16', 'max_gross_x': 1.0609, 'min_equity': 49531.8989, 'worst_mae': 0.1134, 'final_equity': 78167.9731, 'open_minutes': 521251}


...

P2_catastrophe_3x CHENTO_BTC {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.2903, 'at': '2023-10-09 16:29', 'max_gross_x': 3.3868, 'min_equity': 9743.0672, 'worst_mae': 0.1238, 'final_equity': 23456.8198, 'open_minutes': 305303}


P2_catastrophe_3x CHENTO_ETH {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.2392, 'at': '2023-01-08 00:40', 'max_gross_x': 4.0943, 'min_equity': 9047.8244, 'worst_mae': 0.1089, 'final_equity': 13822.6818, 'open_minutes': 229174}


P2_catastrophe_3x SHORT_SQUEEZE {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.1704, 'at': '2023-09-26 16:00', 'max_gross_x': 5.7017, 'min_equity': 9363.4574, 'worst_mae': 0.0572, 'final_equity': 10730.5843, 'open_minutes': 14296}


P2_catastrophe_3x SQUEEZE_BULL {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.9715, 'at': '2022-08-02 09:33', 'max_gross_x': 1.0241, 'min_equity': 9764.7121, 'worst_mae': 0.1043, 'final_equity': 14745.3691, 'open_minutes': 234063}


P2_catastrophe_3x POOLED $50k {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.9561, 'at': '2023-12-03 19:16', 'max_gross_x': 1.0404, 'min_equity': 49470.7141, 'worst_mae': 0.1238, 'final_equity': 72755.4549, 'open_minutes': 691806}
DECISION CHENTO_BTC P1_time_only: {'halves_higher': False, 'mar_higher': False, 'maxdd_ok': True, 'no_liquidation': True, 'safety_2x': True} -> KEEP THE STOP
DECISION CHENTO_BTC P1b_target_only: {'halves_higher': False, 'mar_higher': False, 'maxdd_ok': False, 'no_liquidation': True, 'safety_2x': True} -> KEEP THE STOP
DECISION CHENTO_BTC P2_catastrophe_3x: {'halves_higher': False, 'mar_higher': False, 'maxdd_ok': False, 'no_liquidation': True, 'safety_2x': True} -> KEEP THE STOP
DECISION CHENTO_ETH P1_time_only: {'halves_higher': False, 'mar_higher': False, 'maxdd_ok': False, 'no_liquidation': True, 'safety_2x': False} -> KEEP THE STOP
DECISION CHENTO_ETH P1b_target_only: {'halves_higher': False, 'mar_higher': False, 'maxdd_ok': False, 'no_liquidation

## S3b — POST-HOC sensitivity (not pre-registered): SHORT_SQUEEZE single-open

In [4]:
%run run_s3b_single_open.py

P0_shipped {'n': 71, 'dropped': 0, 'mean_r': 0.481, 'first_half_r': 0.388, 'second_half_r': 0.576, 'mar': 0.191, 'maxdd_pct': -7.881, 'pct_per_year': np.float64(1.504), 'episodes': 0, 'min_distance': 0.329, 'worst_mae': 0.019, 'max_gross_x': 2.992, 'safety_2x': True}


P1_time_only {'n': 59, 'dropped': 12, 'mean_r': 1.131, 'first_half_r': 0.531, 'second_half_r': 1.751, 'mar': 0.632, 'maxdd_pct': -10.825, 'pct_per_year': np.float64(6.839), 'episodes': 0, 'min_distance': 0.288, 'worst_mae': 0.048, 'max_gross_x': 3.418, 'safety_2x': True}


P1b_target_only {'n': 61, 'dropped': 10, 'mean_r': 0.692, 'first_half_r': 0.435, 'second_half_r': 0.957, 'mar': 0.365, 'maxdd_pct': -11.364, 'pct_per_year': np.float64(4.149), 'episodes': 0, 'min_distance': 0.297, 'worst_mae': 0.048, 'max_gross_x': 3.311, 'safety_2x': True}


P2_catastrophe_3x {'n': 65, 'dropped': 6, 'mean_r': 0.398, 'first_half_r': 0.435, 'second_half_r': 0.361, 'mar': 0.083, 'maxdd_pct': -11.92, 'pct_per_year': np.float64(0.987), 'episodes': 0, 'min_distance': 0.315, 'worst_mae': 0.057, 'max_gross_x': 3.129, 'safety_2x': True}


## Tables

In [5]:
import json, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
S1 = json.load(open('results/s1_exit_policies.json', encoding='utf-8'))
for sl_, d in S1['sleeves'].items():
    print(sl_)
    display(pd.DataFrame({p: dict(n=r['n'], mean_r=r['mean_r'], win=r['win'], worst_r=r['worst_r'], h1_r=r['first_half_r'], h2_r=r['second_half_r'],
                                  mean_pct=r['mean_pnl_pct'], mtm_maxdd_pct=r['maxdd_pct'], pct_per_yr=r['pct_per_year'], mar=r['mar'], notional_x=r['mean_notional_x'])
                          for p, r in d.items()}).T.round(3))
display(pd.DataFrame(S1['adx']).T)
S2 = json.load(open('results/s2_sizing_rules.json', encoding='utf-8'))
for sl_, d in S2['sleeves'].items():
    print(sl_); display(pd.DataFrame(d).T.round(2))
print(json.dumps(S2['ruin'], indent=1))
S3 = json.load(open('results/s3_liquidation.json', encoding='utf-8'))
for pol, d in S3['per_bot'].items():
    print(pol); display(pd.DataFrame(d).T.round(4)); print('pooled:', S3['pooled'][pol])
print(json.dumps(S3['decision'], indent=1))


CHENTO_BTC


,n,mean_r,win,worst_r,h1_r,h2_r,mean_pct,mtm_maxdd_pct,pct_per_yr,mar,notional_x
P0_shipped,101.0,0.800,0.406,-1.000,1.301,0.289,1.491,-9.912,26.963,2.720,1.248
P1_time_only,101.0,0.779,0.525,-5.774,1.580,-0.038,1.460,-14.899,25.996,1.745,1.248
P1b_target_only,101.0,0.768,0.535,-5.774,1.388,0.136,1.438,-15.934,26.032,1.634,1.248
P2_catastrophe_3x,101.0,0.714,0.525,-3.000,1.347,0.068,1.332,-15.904,24.107,1.516,1.248


CHENTO_ETH


,n,mean_r,win,worst_r,h1_r,h2_r,mean_pct,mtm_maxdd_pct,pct_per_yr,mar,notional_x
P0_shipped,77.0,0.653,0.429,-1.000,0.979,0.318,1.147,-10.745,16.521,1.538,0.981
P1_time_only,77.0,0.595,0.494,-3.373,1.231,-0.057,0.953,-20.434,13.384,0.655,0.981
P1b_target_only,77.0,0.466,0.506,-3.373,1.005,-0.087,0.773,-21.675,10.977,0.506,0.981
P2_catastrophe_3x,77.0,0.328,0.506,-3.000,0.761,-0.117,0.496,-25.250,7.219,0.286,0.981


SHORT_SQUEEZE


...

pooled: {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.8123253786874942, 'at': '2023-10-08 13:15', 'max_gross_x': 1.223502935398696, 'min_equity': 49470.71412908463, 'worst_mae': 0.1495370321712716, 'final_equity': 82111.58671308352, 'open_minutes': 811734}
P1b_target_only


,breach_minutes,episodes,min_distance,at,max_gross_x,min_equity,worst_mae,final_equity,open_minutes
CHENTO_BTC,0,0,0.286639,2023-10-09 16:44,3.428892,9743.067223,0.123833,24519.344681,326354
CHENTO_ETH,0,0,0.245614,2023-01-08 00:40,3.990193,8886.184288,0.149537,15952.768176,259919
SHORT_SQUEEZE,0,0,0.15915,2023-09-21 14:01,6.091982,9074.400308,0.0834,11644.255833,18229
SQUEEZE_BULL,0,0,0.971471,2022-08-02 09:33,1.024096,9764.712089,0.104293,16004.258201,245936


pooled: {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.8009212033474867, 'at': '2023-10-08 13:15', 'max_gross_x': 1.2408160944846527, 'min_equity': 49470.71412908463, 'worst_mae': 0.1495370321712716, 'final_equity': 78120.62689057698, 'open_minutes': 741475}
P2_catastrophe_3x


,breach_minutes,episodes,min_distance,at,max_gross_x,min_equity,worst_mae,final_equity,open_minutes
CHENTO_BTC,0,0,0.290265,2023-10-09 16:29,3.386788,9743.067223,0.123833,23456.819758,305303
CHENTO_ETH,0,0,0.239242,2023-01-08 00:40,4.094293,9047.824425,0.108876,13822.681797,229174
SHORT_SQUEEZE,0,0,0.170385,2023-09-26 16:00,5.701745,9363.457352,0.057212,10730.584309,14296
SQUEEZE_BULL,0,0,0.971471,2022-08-02 09:33,1.024096,9764.712089,0.104293,14745.369075,234063


pooled: {'breach_minutes': 0, 'episodes': 0, 'min_distance': 0.9561433802365698, 'at': '2023-12-03 19:16', 'max_gross_x': 1.04042749558746, 'min_equity': 49470.71412908463, 'worst_mae': 0.1238325068228307, 'final_equity': 72755.45493974173, 'open_minutes': 691806}
{
 "CHENTO_BTC": {
  "P1_time_only": {
   "halves_higher": false,
   "mar_higher": false,
   "maxdd_ok": true,
   "no_liquidation": true,
   "safety_2x": true,
   "verdict": "KEEP THE STOP",
   "min_distance": 0.28018610950672956,
   "worst_mae": 0.1238325068228307
  },
  "P1b_target_only": {
   "halves_higher": false,
   "mar_higher": false,
   "maxdd_ok": false,
   "no_liquidation": true,
   "safety_2x": true,
   "verdict": "KEEP THE STOP",
   "min_distance": 0.28663940456970244,
   "worst_mae": 0.1238325068228307
  },
  "P2_catastrophe_3x": {
   "halves_higher": false,
   "mar_higher": false,
   "maxdd_ok": false,
   "no_liquidation": true,
   "safety_2x": true,
   "verdict": "KEEP THE STOP",
   "min_distance": 0.290265019